In [17]:
import torch
import torch.nn as nn

import numpy as np

import mne
import matplotlib.pyplot as plt

mne.set_log_level("WARNING") # to suppress info messages

In [18]:
num_features = 40
kernel_size_temporal = 30
kernel_size_spatial = 14
kernel_size_avg_pool = 15
ffn_embedding_dim = 80
vocab_size = 3

temporal_conv = nn.Conv2d(in_channels=1,
                        out_channels=num_features,
                        kernel_size=(1, kernel_size_temporal),
                        padding="same")

spatial_conv = nn.Conv2d(in_channels=num_features,
                                       out_channels=num_features,
                                       kernel_size=(kernel_size_spatial, 1),
                                       padding="valid")

batch_norm1 = nn.BatchNorm2d(num_features)
batch_norm2 = nn.BatchNorm2d(num_features)

elu = nn.ELU()

avg_pool = nn.AvgPool2d(kernel_size=(1, kernel_size_avg_pool))

fc1 = nn.Linear(2400, ffn_embedding_dim)
fc2 = nn.Linear(ffn_embedding_dim, vocab_size)

softmax = nn.Softmax(dim=1)

In [19]:
x = torch.randn(32, 14, 900) # (N, C, T) where H is num_channels

x = x.unsqueeze(1) # (N, C, H, W) where C is 1 and H is num_channels

x = temporal_conv(x) # (32, 40, 14, 900)
x = batch_norm1(x) # (32, 40, 14, 900)
x = elu(x) # (32, 40, 14, 900)
print("before:", x.shape)

x = spatial_conv(x) # (32, 40, 1, 900)
x = batch_norm2(x) # (32, 40, 1, 900)
x = elu(x) # (32, 40, 1, 900)
print("after:", x.shape)

x = avg_pool(x) # (32, 40, 1, 60)
print("pooled:", x.shape)

x = torch.flatten(x, 1) # (32, 40 * 1 * 60) = (32, 2400)
print("flattened:", x.shape)

x = fc1(x)
x = fc2(x)
print("dense:", x.shape)

x = softmax(x) # (32, 3) - each row is a probability distribution across the 3 classes
print("softmax:", x.shape)
print(x)

before: torch.Size([32, 40, 14, 900])
after: torch.Size([32, 40, 1, 900])
pooled: torch.Size([32, 40, 1, 60])
flattened: torch.Size([32, 2400])
dense: torch.Size([32, 3])
softmax: torch.Size([32, 3])
tensor([[0.3596, 0.2806, 0.3597],
        [0.3586, 0.3139, 0.3275],
        [0.3382, 0.3180, 0.3438],
        [0.3933, 0.2843, 0.3224],
        [0.3212, 0.3005, 0.3783],
        [0.3464, 0.3143, 0.3393],
        [0.3821, 0.2985, 0.3194],
        [0.3579, 0.3148, 0.3273],
        [0.3545, 0.2868, 0.3587],
        [0.3686, 0.2922, 0.3392],
        [0.3250, 0.3047, 0.3703],
        [0.3983, 0.3068, 0.2949],
        [0.3655, 0.2759, 0.3586],
        [0.3784, 0.3023, 0.3194],
        [0.3767, 0.2862, 0.3372],
        [0.3886, 0.2934, 0.3180],
        [0.3869, 0.2730, 0.3401],
        [0.3647, 0.3129, 0.3224],
        [0.3797, 0.3020, 0.3183],
        [0.3490, 0.2778, 0.3732],
        [0.4264, 0.2902, 0.2834],
        [0.3912, 0.3038, 0.3050],
        [0.3631, 0.2973, 0.3396],
        [0.3751, 0

In [20]:
raw = mne.io.read_raw_edf("/var/log/thavamount/eeg_dataset/motor_eeg/1.0.0/S001/S001R03.edf", preload=True)

In [21]:
events, event_ids = mne.events_from_annotations(raw)
epochs = mne.Epochs(raw, events=events)

In [22]:
print(raw.get_data().shape) # (64, 20000)
print(events.shape) # every row is [sample, 0, event_id]
print(epochs["2"].get_data().shape) # corresponding to right hand, (n_epochs, C, T)
print(epochs.get_data().shape) # (29, 64, 113)

(64, 20000)
(30, 3)
(8, 64, 113)
(29, 64, 113)


In [23]:
# we want to take the differences between the first column numbers in events to get the time duration of each epoch
event_space = events[1:, 0] - events[:-1, 0]
print(event_space)

[672 656 672 656 672 656 672 656 672 656 672 656 672 656 672 656 672 656
 672 656 672 656 672 656 672 656 672 656 672]


In [24]:
events, event_ids = mne.events_from_annotations(raw)
events = np.delete(events, 1, axis=1) # every row: (sample, event_id)
print(events)
print(epochs.get_data())

[[    0     1]
 [  672     3]
 [ 1328     1]
 [ 2000     2]
 [ 2656     1]
 [ 3328     2]
 [ 3984     1]
 [ 4656     3]
 [ 5312     1]
 [ 5984     3]
 [ 6640     1]
 [ 7312     2]
 [ 7968     1]
 [ 8640     2]
 [ 9296     1]
 [ 9968     3]
 [10624     1]
 [11296     2]
 [11952     1]
 [12624     3]
 [13280     1]
 [13952     3]
 [14608     1]
 [15280     2]
 [15936     1]
 [16608     2]
 [17264     1]
 [17936     3]
 [18592     1]
 [19264     2]]
[[[-1.50000000e-05 -6.00000000e-06  4.00000000e-06 ... -5.40000000e-05
   -4.80000000e-05 -6.00000000e-05]
  [ 2.90909091e-06 -4.09090909e-06  2.90909091e-06 ... -3.70909091e-05
   -3.50909091e-05 -4.70909091e-05]
  [ 3.48484848e-06  9.48484848e-06  6.48484848e-06 ... -2.85151515e-05
   -3.15151515e-05 -4.25151515e-05]
  ...
  [ 1.71818182e-05  1.01818182e-05  1.41818182e-05 ... -1.38181818e-05
   -1.58181818e-05  1.81818182e-07]
  [ 9.03030303e-06 -2.96969697e-06 -6.96969697e-06 ... -4.09696970e-05
   -6.09696970e-05 -4.19696970e-05]
  [-4.48

In [25]:
# checking why some epochs are dropped
print(len(events))
print(len(epochs.events))
print(epochs.drop_log)

30
29
(('NO_DATA',), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), ())


In [26]:
from tqdm import tqdm
from eeg.eeg_data import EEGCNN, HandDatasetCNN

In [48]:
hand_dataset_cnn: HandDatasetCNN = HandDatasetCNN(num_folders=5)
model = EEGCNN(seq_len=hand_dataset_cnn.train_eeg_chunks.shape[-1],
               num_features=10,
               kernel_size_temporal=10,
               kernel_size_spatial=64,
               kernel_size_avg_pool=10,
               ffn_embedding_dim=10
               )

state_dict = torch.load("/var/log/thavamount/eeg_ckpts/cnn/eeg_basic_cnn_epoch_1000.pth", map_location="cuda")
model.load_state_dict(state_dict["model"])

<All keys matched successfully>

In [49]:
eeg_chunk, label_chunk = hand_dataset_cnn.get_validation_data(0)
print(eeg_chunk.shape)
print(label_chunk.shape)

(64, 113)
torch.Size([])


In [50]:
@torch.no_grad()
def inference(
    model: torch.nn.Module, 
    dataset: HandDatasetCNN,
    num_runs: int = 1,
    device: str = "cuda"
) -> torch.Tensor:

    model.to(device)
    model.eval()

    labels = []
    gt = []

    for i in tqdm(range(num_runs)):
        eeg_chunk, label_chunk = dataset.get_validation_data(i)

        eeg_chunk = torch.tensor(eeg_chunk).to(device).float().unsqueeze(0)
        label_chunk = label_chunk.to(device).unsqueeze(0)

        output = model(eeg_chunk).squeeze(0)

        label = torch.argmax(output).item()
        labels.append(label)
        gt.append(label_chunk.item())

    return labels, gt

In [52]:
labels, gt = inference(model, hand_dataset_cnn, num_runs=100)
print("----------labels----------")
print(labels)
print("----------ground truth----------")
print(gt)
print()
print(f"accuracy: {(np.array(labels) == np.array(gt)).mean() * 100}%")

100%|██████████| 100/100 [00:00<00:00, 653.87it/s]

----------labels----------
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
----------ground truth----------
[1, 0, 2, 0, 2, 0, 1, 0, 2, 0, 1, 0, 2, 0, 1, 0, 2, 0, 1, 0, 1, 0, 2, 0, 1, 0, 2, 0, 2, 2, 0, 1, 0, 1, 0, 2, 0, 2, 0, 1, 0, 1, 0, 2, 0, 1, 0, 2, 0, 2, 0, 1, 0, 1, 0, 2, 0, 2, 2, 0, 1, 0, 1, 0, 2, 0, 1, 0, 2, 0, 2, 0, 1, 0, 2, 0, 1, 0, 2, 0, 1, 0, 1, 0, 2, 0, 1, 2, 0, 1, 0, 2, 0, 1, 0, 1, 0, 2, 0, 2]

accuracy: 48.0%
